# Linear Regression Training Pipeline

This notebook implements an end-to-end Linear Regression training pipeline with:
- Data loading and exploration
- Preprocessing and scaling
- Train-test split
- Model training
- Hyperparameter tuning with Optuna
- Model evaluation and validation

## 1. Import Required Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib

# Modeling
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, RobustScaler
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
# Mute warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

Libraries imported successfully.


C:\Users\racho\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load and Explore Dataset

In [4]:
DATA_PATH = '../../data/typhoon_impact_with_extreme_weather.csv' 

# Lead Developer's Configuration
INPUT_FEATURES = [
    'max_sustained_wind_kph',
    'typhoon_type',
    'max_24hr_rainfall_mm',
    'total_storm_rainfall_mm',
    'min_pressure_hpa'
]

# Ordinal Mapping (Better for Linear Regression)
TYPHOON_TYPE_MAPPING = {
    'TD': 0,   'TS': 1,   'STS': 2,  'TY': 3,   'STY': 4
}

try:
    df = pd.read_csv(DATA_PATH)
    
    # Normalize Column Names
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('/', '_')
    
    # Target Selection
    TARGET_COLUMN = 'cost' 

    print(f"Data Loaded. Shape: {df.shape}")
    print(f"Target Column: {TARGET_COLUMN}")
    
    # Drop rows where target is missing
    df = df.dropna(subset=[TARGET_COLUMN])
    
    # Check for Zero-Inflation
    zeros = (df[TARGET_COLUMN] == 0).sum()
    print(f"Zero-Cost Events: {zeros} ({zeros/len(df):.1%})")

except Exception as e:
    print(f"Error: {e}")

Data Loaded. Shape: (1776, 30)
Target Column: cost
Zero-Cost Events: 1333 (75.1%)


## 3. Data Preprocessing

In [13]:
def preprocess_features(X, fit=False, imputer=None, scaler=None, poly=None):
    """Apply all preprocessing steps"""
    X_proc = X.copy()
    
    # Ordinal encoding
    X_proc['typhoon_type'] = X_proc['typhoon_type'].astype(str).str.upper().map(TYPHOON_TYPE_MAPPING)
    X_proc['typhoon_type'] = X_proc['typhoon_type'].fillna(X_proc['typhoon_type'].median())
    
    # Physics-based features (BEFORE polynomial expansion)
    X_proc['wind_power'] = X_proc['max_sustained_wind_kph'] ** 3
    X_proc['pressure_drop'] = 1013 - X_proc['min_pressure_hpa']
    
    # CRITICAL: Wind-Rain interaction (key damage indicator)
    X_proc['wind_rain_interaction'] = (
        X_proc['max_sustained_wind_kph'] * X_proc['total_storm_rainfall_mm']
    )
    
    # Imputation
    if fit:
        imputer = SimpleImputer(strategy='median')
        X_imputed = pd.DataFrame(
            imputer.fit_transform(X_proc), 
            columns=X_proc.columns, 
            index=X_proc.index
        )
    else:
        X_imputed = pd.DataFrame(
            imputer.transform(X_proc), 
            columns=X_proc.columns, 
            index=X_proc.index
        )
    
    # Polynomial features (degree 2 for interactions)
    if fit:
        poly = PolynomialFeatures(degree=2, include_bias=False)
        X_poly_array = poly.fit_transform(X_imputed)
    else:
        X_poly_array = poly.transform(X_imputed)
    
    feature_names = poly.get_feature_names_out(X_imputed.columns)
    X_poly = pd.DataFrame(X_poly_array, columns=feature_names, index=X_imputed.index)
    
    # Scaling
    if fit:
        scaler = RobustScaler()
        X_scaled = pd.DataFrame(
            scaler.fit_transform(X_poly),
            columns=X_poly.columns,
            index=X_poly.index
        )
    else:
        X_scaled = pd.DataFrame(
            scaler.transform(X_poly),
            columns=X_poly.columns,
            index=X_poly.index
        )
    
    if fit:
        return X_scaled, imputer, scaler, poly
    else:
        return X_scaled

# Apply preprocessing
X_train_scaled, imputer, scaler, poly = preprocess_features(X_train_raw, fit=True)
X_test_scaled = preprocess_features(X_test_raw, fit=False, imputer=imputer, scaler=scaler, poly=poly)

print(f"Features after preprocessing: {X_train_scaled.shape[1]}")

Features after preprocessing: 44


## 4. Train-Test Split

In [14]:
X_raw = df[INPUT_FEATURES].copy()
y = df[TARGET_COLUMN].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=(y > 0).astype(int)
)

print(f"Train size: {len(X_train_raw)}, Test size: {len(X_test_raw)}")

Train size: 1420, Test size: 356


## 6. Basic Linear Regression Model Training

In [7]:
print("\n=== Training Damage Classifier ===")

y_train_binary = (y_train > 0).astype(int)
y_test_binary = (y_test > 0).astype(int)

# Train base classifier
base_clf = LogisticRegression(
    class_weight='balanced',
    C=0.1,  # Add regularization
    random_state=42,
    max_iter=1000
)

# CRITICAL FIX: Calibrate probabilities for better thresholding
clf = CalibratedClassifierCV(base_clf, cv=3, method='sigmoid')
clf.fit(X_train_scaled, y_train_binary)

# Evaluate classifier
y_train_pred_binary = clf.predict(X_train_scaled)
y_test_pred_binary = clf.predict(X_test_scaled)

print(f"\nClassifier Performance:")
print(f"Train Accuracy: {accuracy_score(y_train_binary, y_train_pred_binary):.2%}")
print(f"Test Accuracy: {accuracy_score(y_test_binary, y_test_pred_binary):.2%}")
print(f"\nTest Set Classification Report:")
print(classification_report(y_test_binary, y_test_pred_binary, target_names=['No Damage', 'Damage']))


=== Training Damage Classifier ===

Classifier Performance:
Train Accuracy: 79.08%
Test Accuracy: 77.81%

Test Set Classification Report:
              precision    recall  f1-score   support

   No Damage       0.78      0.99      0.87       267
      Damage       0.78      0.16      0.26        89

    accuracy                           0.78       356
   macro avg       0.78      0.57      0.57       356
weighted avg       0.78      0.78      0.72       356



In [8]:
print("\n=== Optimizing Classification Threshold ===")

# Create validation split from training data
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=(y_train > 0).astype(int)
)

# Get probabilities on validation set
probs_val = clf.predict_proba(X_val_opt)[:, 1]
y_val_binary = (y_val_opt > 0).astype(int)

# Find optimal threshold
best_threshold = 0.5
best_f1 = 0

for threshold in np.arange(0.1, 0.9, 0.05):
    preds = (probs_val >= threshold).astype(int)
    
    # Calculate F1 score (better than accuracy for imbalanced data)
    tp = ((preds == 1) & (y_val_binary == 1)).sum()
    fp = ((preds == 1) & (y_val_binary == 0)).sum()
    fn = ((preds == 0) & (y_val_binary == 1)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"Optimal Threshold: {best_threshold:.2f} (F1: {best_f1:.3f})")


=== Optimizing Classification Threshold ===
Optimal Threshold: 0.25 (F1: 0.524)


## 7. Hyperparameter Tuning with Optuna

In [9]:
print("\n=== Training Cost Regressor ===")

mask_nonzero = y_train > 0
X_train_nonzero = X_train_scaled[mask_nonzero]
y_train_nonzero = y_train[mask_nonzero]

# Log transform for stability
y_train_log = np.log1p(y_train_nonzero)

# Optuna optimization
def objective(trial):
    alpha = trial.suggest_float('alpha', 0.01, 100.0, log=True)
    model_type = trial.suggest_categorical('model_type', ['ridge', 'lasso'])
    
    if model_type == 'ridge':
        model = Ridge(alpha=alpha, random_state=42)
    else:
        model = Lasso(alpha=alpha, random_state=42, max_iter=2000)
    
    scores = cross_val_score(
        model, X_train_nonzero, y_train_log,
        cv=3, scoring='neg_mean_squared_error'
    )
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=False)

best_params = study.best_params
print(f"Best Regressor Params: {best_params}")

# Train final regressor
if best_params['model_type'] == 'ridge':
    regressor = Ridge(alpha=best_params['alpha'], random_state=42)
else:
    regressor = Lasso(alpha=best_params['alpha'], random_state=42, max_iter=2000)

regressor.fit(X_train_nonzero, y_train_log)


[I 2025-12-11 14:18:16,388] A new study created in memory with name: no-name-80f3e82b-4ea0-4ee9-97b6-4ae28f3ce207
[I 2025-12-11 14:18:16,401] Trial 0 finished with value: -3.752040740400157 and parameters: {'alpha': 70.3674020972506, 'model_type': 'ridge'}. Best is trial 0 with value: -3.752040740400157.
[I 2025-12-11 14:18:16,414] Trial 1 finished with value: -3.6388445591851624 and parameters: {'alpha': 1.188703158409454, 'model_type': 'ridge'}. Best is trial 1 with value: -3.6388445591851624.
[I 2025-12-11 14:18:16,424] Trial 2 finished with value: -3.632288534396673 and parameters: {'alpha': 1.3297217949129296, 'model_type': 'ridge'}. Best is trial 2 with value: -3.632288534396673.
[I 2025-12-11 14:18:16,434] Trial 3 finished with value: -3.6627919961152515 and parameters: {'alpha': 22.705373962136505, 'model_type': 'ridge'}. Best is trial 2 with value: -3.632288534396673.
[I 2025-12-11 14:18:16,444] Trial 4 finished with value: -4.9676593998203735 and parameters: {'alpha': 3.94701


=== Training Cost Regressor ===


[I 2025-12-11 14:18:16,593] Trial 16 finished with value: -3.632336777677011 and parameters: {'alpha': 14.152728268053378, 'model_type': 'ridge'}. Best is trial 15 with value: -3.601211753745069.
[I 2025-12-11 14:18:16,605] Trial 17 finished with value: -3.59822259857838 and parameters: {'alpha': 4.59036938403192, 'model_type': 'ridge'}. Best is trial 17 with value: -3.59822259857838.
[I 2025-12-11 14:18:16,618] Trial 18 finished with value: -4.9676593998203735 and parameters: {'alpha': 4.950358172422558, 'model_type': 'lasso'}. Best is trial 17 with value: -3.59822259857838.
[I 2025-12-11 14:18:16,630] Trial 19 finished with value: -3.7885417650316042 and parameters: {'alpha': 0.17223588351695227, 'model_type': 'ridge'}. Best is trial 17 with value: -3.59822259857838.
[I 2025-12-11 14:18:16,641] Trial 20 finished with value: -3.6045714777896816 and parameters: {'alpha': 7.218804132847099, 'model_type': 'ridge'}. Best is trial 17 with value: -3.59822259857838.
[I 2025-12-11 14:18:16,65

Best Regressor Params: {'alpha': 4.38044438724685, 'model_type': 'ridge'}


,alpha,4.38044438724685
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,42


In [10]:
print("Best Hyperparameters:")
print(best_params)

Best Hyperparameters:
{'alpha': 4.38044438724685, 'model_type': 'ridge'}


## 8. Model Evaluation and Validation

In [11]:
print("\n=== Final Evaluation ===")

# Get classifier probabilities
probs_test = clf.predict_proba(X_test_scaled)[:, 1]
pred_is_damage = (probs_test >= best_threshold).astype(int)

# Get regressor predictions (in log space)
pred_log = regressor.predict(X_test_scaled)
pred_amount = np.expm1(pred_log)
pred_amount = np.maximum(pred_amount, 0)  # Clip negatives

# Combine: only predict cost if classifier says damage
final_predictions = pred_is_damage * pred_amount

# Calculate metrics
acc = accuracy_score(y_test_binary, pred_is_damage)
mae = mean_absolute_error(y_test, final_predictions)
r2 = r2_score(y_test, final_predictions)

# Additional metric: MAE on non-zero cases only
mask_test_nonzero = y_test > 0
if mask_test_nonzero.sum() > 0:
    mae_nonzero = mean_absolute_error(
        y_test[mask_test_nonzero],
        final_predictions[mask_test_nonzero]
    )
else:
    mae_nonzero = np.nan

print(f"\nOverall Performance:")
print(f"Damage Detection Accuracy: {acc:.2%}")
print(f"MAE (All Cases): ${mae:,.2f}")
print(f"MAE (Non-Zero Only): ${mae_nonzero:,.2f}")
print(f"R² Score: {r2:.4f}")


=== Final Evaluation ===

Overall Performance:
Damage Detection Accuracy: 66.57%
MAE (All Cases): $305,301.35
MAE (Non-Zero Only): $876,339.60
R² Score: 0.1604


## 10. Model Output and Summary

In [12]:
results_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': final_predictions,
    'Probability': probs_test,
    'Classified_As_Damage': pred_is_damage
})

results_df['Error'] = results_df['Actual'] - results_df['Predicted']
results_df['Abs_Error'] = results_df['Error'].abs()
results_df['Pct_Error'] = np.where(
    results_df['Actual'] > 0,
    (results_df['Abs_Error'] / results_df['Actual']) * 100,
    0
)

print("\n=== Sample Predictions ===")
print(results_df.head(20).to_string())

# Confusion matrix analysis
print("\n=== Classification Analysis ===")
tp = ((pred_is_damage == 1) & (y_test_binary == 1)).sum()
fp = ((pred_is_damage == 1) & (y_test_binary == 0)).sum()
tn = ((pred_is_damage == 0) & (y_test_binary == 0)).sum()
fn = ((pred_is_damage == 0) & (y_test_binary == 1)).sum()

print(f"True Positives (Correctly predicted damage): {tp}")
print(f"False Positives (Predicted damage, none occurred): {fp}")
print(f"True Negatives (Correctly predicted no damage): {tn}")
print(f"False Negatives (Missed damage events): {fn}")


=== Sample Predictions ===
      Actual     Predicted  Probability  Classified_As_Damage         Error     Abs_Error    Pct_Error
0        0.0  0.000000e+00     0.107031                     0  0.000000e+00  0.000000e+00     0.000000
1   739040.0  3.494016e+05     0.272425                     1  3.896384e+05  3.896384e+05    52.722234
2        0.0  0.000000e+00     0.158226                     0  0.000000e+00  0.000000e+00     0.000000
3   180000.0  3.494016e+05     0.272425                     1 -1.694016e+05  1.694016e+05    94.112002
4        0.0  0.000000e+00     0.166891                     0  0.000000e+00  0.000000e+00     0.000000
5   491042.3  2.846493e+05     0.320419                     1  2.063930e+05  2.063930e+05    42.031610
6   544600.0  9.623826e+04     0.359822                     1  4.483617e+05  4.483617e+05    82.328634
7        0.0  0.000000e+00     0.170148                     0  0.000000e+00  0.000000e+00     0.000000
8        0.0  0.000000e+00     0.113006      

In [20]:
results_df.to_csv('predictions_improved_pipeline.csv', index=False)

# joblib.dump({
#     'imputer': imputer,
#     'scaler': scaler,
#     'poly': poly,
#     'classifier': clf,
#     'regressor': regressor,
#     'threshold': best_threshold
# }, 'improved_lr_pipeline.joblib')

print("Models and results saved successfully!")

Models and results saved successfully!
